[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jcomajuncosas/tp2/blob/main/05_sintesi/sessio08_oscil_ladors_classes/recursos/patches_bloc5a.ipynb)

# Patches — Bloc 5a, Sessió 8
### Síntesi amb classes: Oscillator, Envelope, SamplePlayer

Notebook de la **demo guiada**. Executa les cel·les en ordre i escolta cada resultat.

In [ ]:
import numpy as np
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import urllib.request

sample_rate = 44100

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/05_sintesi/sessio08_oscil_ladors_classes/recursos/audio/"
urllib.request.urlretrieve(base_url + "kick_sample.wav", "kick_sample.wav")
print("Fitxers carregats")

## 1. Un oscil·lador sense memòria

Primer, l'oscil·lador "com el faríem amb el que ja sabem": una classe que només guarda la freqüència. Cada crida a `generate()` torna a començar a `t = 0`.

In [ ]:
class OscillatorSenseMemoria:
    def __init__(self, freq=440.0, sample_rate=44100):
        self.freq = freq
        self.sample_rate = sample_rate

    def generate(self, duration):
        n = int(round(self.sample_rate * duration))
        t = np.arange(n) / self.sample_rate          # ⚠️ t torna a començar a 0 a CADA crida
        return np.sin(2 * np.pi * self.freq * t)


osc = OscillatorSenseMemoria(freq=440)
print("2 segons d'una tirada:")
display(Audio(osc.generate(2.0), rate=sample_rate))

# I ara tros a tros, com ho demanarà un callback: 200 trossos de 512 mostres
trossos = []
for i in range(200):
    trossos.append(osc.generate(512 / sample_rate))
sense = np.concatenate(trossos)
print("Els mateixos ~2 segons, en 200 trossos enganxats:")
display(Audio(sense, rate=sample_rate))

## 2. La fase, dins de l'objecte

A la S05 la solució era una global: `fase_sinus`. I per a cada oscil·lador nou, una global nova (`fase_ringmod`...). Ara la fase la guarda **l'objecte**: `self._phase`. Són les mateixes dues línies de la S05.

El guió baix (`_phase`) és un conveni: *"estat intern, no el toquis des de fora"*. Python no ho impedeix; altres llenguatges (C++, Java) ho fan complir amb `private`.

In [ ]:
class Oscillator:
    def __init__(self, freq=440.0, waveform='sine', sample_rate=44100):
        self.freq = freq                # paràmetres: es poden canviar des de fora
        self.waveform = waveform
        self.sample_rate = sample_rate
        self._phase = 0.0               # estat intern: on som del cicle (0 … 2π)

    def generate(self, duration):
        n = int(round(self.sample_rate * duration))
        increment = 2 * np.pi * self.freq / self.sample_rate
        fases = self._phase + increment * np.arange(n)      # continuem on ho vam deixar

        # recordem on ha de començar la PROPERA crida (com fase_sinus a la S05)
        self._phase = (self._phase + increment * n) % (2 * np.pi)

        if self.waveform == 'sine':
            return np.sin(fases)
        elif self.waveform == 'square':
            return signal.square(fases)
        elif self.waveform == 'sawtooth':
            return signal.sawtooth(fases)
        else:
            print("Forma d'ona desconeguda:", self.waveform, "→ retorno silenci")
            return np.zeros(n)

In [ ]:
osc = Oscillator(freq=440)
trossos = []
for i in range(200):
    trossos.append(osc.generate(512 / sample_rate))
amb = np.concatenate(trossos)
print("200 trossos, ara AMB la fase guardada:")
display(Audio(amb, rate=sample_rate))

# Zoom a la frontera entre el tros 1 i el tros 2 (mostra 512)
zona = np.arange(400, 624)
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(zona, sense[zona])
axes[0].axvline(512, color='red', linestyle='--')
axes[0].set_title("SENSE memòria: salt a la frontera")
axes[1].plot(zona, amb[zona])
axes[1].axvline(512, color='red', linestyle='--')
axes[1].set_title("AMB self._phase: continu")
axes[1].set_xlabel("mostra")
plt.tight_layout()
plt.show()

## 3. Dues instàncies, dues memòries

Cada `Oscillator()` porta la seva freqüència i la seva fase. Si en volem cinc, en creem cinc: cap global nova.

In [ ]:
osc_a = Oscillator(freq=440)
osc_b = Oscillator(freq=660, waveform='square')

osc_a.generate(0.01)
osc_a.generate(0.01)
osc_b.generate(0.01)

print("osc_a: freq =", osc_a.freq, "  _phase =", round(osc_a._phase, 3))
print("osc_b: freq =", osc_b.freq, "  _phase =", round(osc_b._phase, 3))

# Canviar la configuració: directament a l'atribut (res de set_freq)
osc_a.freq = 880
print("osc_a ara:", osc_a.freq, "Hz — osc_b no s'ha mogut:", osc_b.freq, "Hz")

## 4. La classe `Envelope` (ADSR)

In [ ]:
class Envelope:
    def __init__(self, attack=0.01, decay=0.1, sustain=0.7, release=0.2):
        self.attack = attack
        self.decay = decay
        self.sustain = sustain          # és un NIVELL (0-1), no un temps
        self.release = release

    def generate(self, note_duration, sample_rate=44100):
        """note_duration: temps que la nota està premuda (el release s'afegeix DESPRÉS)."""
        n_attack  = int(self.attack  * sample_rate)
        n_decay   = int(self.decay   * sample_rate)
        n_sustain = max(0, int(note_duration * sample_rate) - n_attack - n_decay)
        n_release = int(self.release * sample_rate)

        env_attack  = np.linspace(0, 1, n_attack)
        env_decay   = np.linspace(1, self.sustain, n_decay)
        env_sustain = np.full(n_sustain, self.sustain)
        env_release = np.linspace(self.sustain, 0, n_release)

        return np.concatenate([env_attack, env_decay, env_sustain, env_release])

In [ ]:
env = Envelope(attack=0.05, decay=0.15, sustain=0.6, release=0.3)
corba = env.generate(note_duration=0.5)

plt.figure(figsize=(10, 3))
plt.plot(np.arange(len(corba)) / sample_rate, corba)
plt.title("Envolvent ADSR (nota premuda 0.5 s + release 0.3 s)")
plt.xlabel("temps (s)")
plt.ylabel("amplitud")
plt.show()

## 5. Oscil·lador × envolvent

L'oscil·lador dona el *timbre*; l'envolvent, la *forma temporal*. Es combinen multiplicant. Generem tantes mostres d'oscil·lador com en té la corba.

In [ ]:
osc = Oscillator(freq=440, waveform='sine')
env = Envelope(attack=0.02, decay=0.1, sustain=0.6, release=0.3)

corba = env.generate(note_duration=0.5)
ona = osc.generate(len(corba) / sample_rate)
nota_final = ona * corba

plt.figure(figsize=(10, 3))
plt.plot(nota_final)
plt.title("Oscillator × Envelope — ara SÍ té forma de nota")
plt.show()

print("Sense envolvent:")
display(Audio(ona, rate=sample_rate))
print("Amb envolvent:")
display(Audio(nota_final, rate=sample_rate))

## 6. MIDI → freqüència

In [ ]:
def note_to_freq(note_number):
    """Número MIDI → freqüència (Hz). Ja la coneixes del Bloc 4."""
    return 440.0 * (2 ** ((note_number - 69) / 12))


for nota in [60, 64, 67, 72]:
    print("Nota MIDI", nota, "→", round(note_to_freq(nota), 2), "Hz")

## 7. Una funció que toca una nota MIDI completa

In [ ]:
def tocar_nota(note_number, note_duration, waveform='sine',
               attack=0.02, decay=0.1, sustain=0.6, release=0.2):
    osc = Oscillator(freq=note_to_freq(note_number), waveform=waveform)
    env = Envelope(attack=attack, decay=decay, sustain=sustain, release=release)
    corba = env.generate(note_duration)
    ona = osc.generate(len(corba) / sample_rate)
    return ona * corba


# Un petit arpegi (la idea del Bloc 4)
acord = [60, 64, 67, 72]
trossos = []
for nota in acord:
    trossos.append(tocar_nota(nota, note_duration=0.2))

Audio(np.concatenate(trossos), rate=sample_rate)

## 8. La classe `SamplePlayer`: un component diferent

L'`Oscillator` **genera** so amb matemàtiques; el `SamplePlayer` **reprodueix** un so ja gravat. `generate(duration)` retorna exactament `duration` segons: retalla el sample o hi afegeix silenci.

In [ ]:
class SamplePlayer:
    def __init__(self, filepath, gain=1.0, sample_rate=44100):
        self.gain = gain
        self.sample_rate = sample_rate
        data, sr = sf.read(filepath)
        if sr != sample_rate:
            print("Atenció: el fitxer és a", sr, "Hz i esperàvem", sample_rate, "Hz")
        self.sample = data

    def generate(self, duration):
        n = int(round(self.sample_rate * duration))
        if n <= len(self.sample):
            return self.sample[:n] * self.gain              # el pas és més curt: retallem
        silenci = np.zeros(n - len(self.sample))
        return np.concatenate([self.sample, silenci]) * self.gain   # sobra temps: silenci

In [ ]:
kick = SamplePlayer('kick_sample.wav')

print("Oscillator (genera matemàticament):")
display(Audio(tocar_nota(36, note_duration=0.2), rate=sample_rate))

print("SamplePlayer (reprodueix un so gravat):")
display(Audio(kick.generate(0.5), rate=sample_rate))

print("Mostres del fitxer:", len(kick.sample), "→ 0.5 s són", len(kick.generate(0.5)), "mostres")

## 9. Mateix nom per a la mateixa feina

`Oscillator.generate(duration)` i `SamplePlayer.generate(duration)` fan la mateixa mena de feina: retornar `duration` segons de so. Com que es diuen igual, una llista pot barrejar-los i el bucle no necessita saber qui és qui.

In [ ]:
baix = Oscillator(55, 'sawtooth')
kick = SamplePlayer('kick_sample.wav', gain=0.9)
fonts = [baix, kick]

pas = 0.25                                  # una corxera a 120 bpm
compas = []
for i in range(8):
    tros = np.zeros(int(round(pas * sample_rate)))
    for font in fonts:
        tros = tros + font.generate(pas)    # el mateix mètode per a tots
    compas.append(tros)

so = np.concatenate(compas)
so = so / np.max(np.abs(so))
Audio(so, rate=sample_rate)
# El baix sona continu entre passos (gràcies a _phase); el kick es dispara a cada pas.